<a href="https://colab.research.google.com/github/Buket12345/climbing-ifsc-dsa210/blob/main/data_collection_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns


github_link = "https://raw.githubusercontent.com/Buket12345/climbing-ifsc-dsa210/main"
info_link = f"{github_link}/athlete_information.csv"
results_link = f"{github_link}/athlete_results.csv"
gdp_link = f"{github_link}/gdp-per-capita-worldbank.csv"
info = pd.read_csv(info_link)
results = pd.read_csv(results_link)
gdp = pd.read_csv(gdp_link)

print("Info:", info.shape)
print("Results:", results.shape)
print("GDP:", gdp.shape)


Info: (16258, 10)
Results: (135444, 8)
GDP: (7063, 4)


In [30]:
ifsc = results.merge(info, on="athlete_id", how="left") #merged 2 datasets and stores the resulting dataset in ifsc
print("Merged IFSC:", ifsc.shape)

Merged IFSC: (135444, 17)


In [31]:

gdp = gdp.rename(columns={
    "Code": "country",
    "Year": "season",
    "GDP per capita, PPP (constant 2021 international $)": "gdp_per_capita"
})

gdp = gdp[["country", "season", "gdp_per_capita"]]

ifsc["season"] = ifsc["season"].astype(int)
gdp["season"] = gdp["season"].astype(int)

print(gdp.head())

  country  season  gdp_per_capita
0     AFG    2000       1617.8264
1     AFG    2001       1454.1108
2     AFG    2002       1774.3087
3     AFG    2003       1815.9282
4     AFG    2004       1776.9182


In [38]:
merged_df = ifsc.merge(gdp, on=["country", "season"], how="left")

# We are checking for unmatched GDP values
missing_gdp = merged_df['gdp_per_capita'].isna().sum()
total_rows = len(merged_df)

print("Missing GDP values:", missing_gdp)
print("Percentage:", (missing_gdp / total_rows) * 100)



Missing GDP values: 9003
Percentage: 6.647027553822982


After merging the IFSC competition data with the World Bank GDP dataset, approximately 27% of rows initially contained missing GDP values. This was mostly caused by a mismatch between the country codes used by the IFSC and the ISO-3 country codes used by the World Bank GDP data set. Therefore, a mapping dictionary was constructed before merging the two datasets.

In [39]:

country_map = {
    "GER": "DEU", "SUI": "CHE", "SLO": "SVN", "NED": "NLD",
    "BUL": "BGR", "INA": "IDN", "IRI": "IRN", "CHI": "CHL",
    "CRO": "HRV", "TPE": "TWN", "DEN": "DNK", "RSA": "ZAF",
    "POR": "PRT", "GRE": "GRC", "LAT": "LVA", "MAS": "MYS",
    "PHI": "PHL", "MGL": "MNG", "GUA": "GTM", "PUR": "PRI",
    "KSA": "SAU", "ESA": "SLV", "CAM": "KHM", "CRC": "CRI",
    "HON": "HND", "MRI": "MUS", "NEP": "NPL", "VEN": "VEN",
    "New Caledonia": "NCL"
}

ifsc["country"] = ifsc["country"].replace(country_map)


After applying this standardization, the proportion of missing GDP values was reduced to approximately 1%.
In addition, the World Bank GDP dataset only provides values up to the year 2023, whereas the IFSC competition dataset includes events from 2024 as well. This mismatch resulted in additional NaN GDP values and since no reliable and finalized GDP per capita data for 2024 was available, the analysis was restricted to competition results from 1991 to 2023.

In [37]:
final_df = ifsc.merge(gdp, on=["country", "season"], how="left")

final_df = final_df[final_df["season"] <= 2023]

print(final_df.shape)
print("Missing GDP:", final_df["gdp_per_capita"].isna().sum())

(127730, 18)
Missing GDP: 1289


In [40]:
analysis_df = final_df.dropna(subset=["gdp_per_capita", "rank"])

print(analysis_df.shape)

analysis_df.to_csv("final_dataset.csv", index=False)

(126441, 18)
